In [3]:
import sys, os

import numpy as np
# Note that the path ./Albedo_project/data_processing is now in the path
import xarray as xr
from cloudpathlib import AnyPath
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Local Imports
curdir = os.getcwd()
print(curdir+"../../data_processing")
sys.path.insert(0, curdir+"/../../data_processing")
sys.path.insert(0, curdir+"/../../visualization")
import ceres_ebaf
import ceres_ebaf_plotting

fig_path = AnyPath("../../../Figs")
current_month_folder="24H1"
ceres_base_path = AnyPath("/Users/mawa7160/dev/data/CERES/EBAF/")
merra_base_path = AnyPath("/Users/mawa7160/dev/data/MERRA2/23Aug")

In [8]:
ebaf_data = xr.open_dataset(ceres_base_path / "24Feb" / "CERES_EBAF-TOA_Ed4.2_Subset_200003-202311.nc")
merra_data = xr.open_dataset(merra_base_path / "Merra2_monthly_01_00-06_23.nc")

In [18]:
sw_clear_sky_ebaf = ceres_ebaf.create_hemisphere_data(ebaf_data["toa_sw_clr_c_mon"])
suf_temp_merra_data = merra_surface_temp_run = ceres_ebaf.create_hemisphere_data(merra_data["TLML"])

In [35]:
sw_anomalie = sw_clear_sky_ebaf.mean()-sw_clear_sky_ebaf
suf_temp_anomalie = suf_temp_merra_data.mean()-suf_temp_merra_data

In [36]:
fig, axis = plt.subplots(1, 3, figsize=(20,10))
axis[0].plot(sw_clear_sky_ebaf.time, sw_clear_sky_ebaf["global"],"b-x")
axis2 = axis[0].twinx()
axis2.plot(suf_temp_merra_data.time, suf_temp_merra_data["global"], "r-x")
plt.title("Global")

axis[1].plot(sw_clear_sky_ebaf.time, sw_clear_sky_ebaf["nh"],"b-x")
axis2 = axis[1].twinx()
axis2.plot(suf_temp_merra_data.time, suf_temp_merra_data["nh"], "r-x")
plt.title("NH")

axis[2].plot(sw_clear_sky_ebaf.time, sw_clear_sky_ebaf["sh"],"b-x")
axis2 = axis[2].twinx()
axis2.plot(suf_temp_merra_data.time, suf_temp_merra_data["sh"], "r-x")
plt.title("SH")


In [37]:
# Plotting the anomalies
fig, axis = plt.subplots(1, 3, figsize=(20,10))
axis[0].plot(sw_anomalie.time, sw_anomalie["global"],"b-x")
axis2 = axis[0].twinx()
axis2.plot(suf_temp_anomalie.time, suf_temp_anomalie["global"], "r-x")
plt.title("Global")
    
axis[1].plot(sw_anomalie.time, sw_anomalie["nh"],"b-x")
axis2 = axis[1].twinx()
axis2.plot(suf_temp_anomalie.time, suf_temp_anomalie["nh"], "r-x")
plt.title("NH")

axis[2].plot(sw_anomalie.time, sw_anomalie["sh"],"b-x")
axis2 = axis[2].twinx()
axis2.plot(suf_temp_anomalie.time, suf_temp_anomalie["sh"], "r-x")
plt.title("SH")

In [43]:
# Calculate the correlation between the anomalies for global, nh, and sh
correlation_global = np.corrcoef(sw_anomalie["global"], suf_temp_anomalie["global"])
correlation_nh = np.corrcoef(sw_anomalie["nh"], suf_temp_anomalie["nh"])
correlation_sh = np.corrcoef(sw_anomalie["sh"], suf_temp_anomalie["sh"])

# Calculate the linear regression for the anomalies for global, nh, and sh
regression_global = np.polyfit(sw_anomalie["global"], suf_temp_anomalie["global"], 1)
regression_nh = np.polyfit(sw_anomalie["nh"], suf_temp_anomalie["nh"], 1)
regression_sh = np.polyfit(sw_anomalie["sh"], suf_temp_anomalie["sh"], 1)

# calculate the correlation c

In [108]:
# Plot the correlation as a scatter plot with the line of best fit
# Print the correlation coefficient and the linear regression
# Print the slope of the linear regression
fig, axis = plt.subplots(1, 3, figsize=(20,10))
axis[0].scatter(sw_anomalie["global"], suf_temp_anomalie["global"])
#axis[0].scatter(sw_clear_sky_ebaf["global"], suf_temp_merra_data["global"])
axis[0].plot(sw_anomalie["global"], regression_global[0]*sw_anomalie["global"]+regression_global[1], "r")
print("Global R-Squared: ", correlation_global[0,1]**2)
print("Global Correlation: ", correlation_global[0,1])
axis[0].set_title("Global")

axis[1].scatter(sw_anomalie["nh"], suf_temp_anomalie["nh"])
axis[1].plot(sw_anomalie["nh"], regression_nh[0]*sw_anomalie["nh"]+regression_nh[1], "r")
print("NH R-Squared: ", correlation_nh[0,1]**2)
print("NH Correlation: ", correlation_nh[0,1])
axis[1].set_title("NH")

axis[2].scatter(sw_anomalie["sh"], suf_temp_anomalie["sh"])
axis[2].plot(sw_anomalie["sh"], regression_sh[0]*sw_anomalie["sh"]+regression_sh[1], "r")
print("SH R-Squared: ", correlation_sh[0,1]**2)
print("SH Correlation: ", regression_sh[0])
plt.title("SH")



# Spatial Correlation Mapping

In [68]:
# For the northern hemisphere in each grid cell calculate the yearly average of the sw clear sky and the surface temperature
sw_yearly_average = ceres_ebaf.apply_time_averaging(ebaf_data["toa_sw_clr_c_mon"])
suf_temp_yearly_average = ceres_ebaf.apply_time_averaging(merra_data["TLML"])

# For each location calculate the correlation between the sw clear sky and the surface temperature


# Calculate the yearly average of the sw clear sky and the surface temperature using a groupby and mean
#sw_yearly_average_groupby = sw_clear_sky_ebaf.groupby("time.year").mean()
#suf_temp_yearly_average_groupby = suf_temp_merra_data.groupby("time.year").mean()


In [109]:
# create a function that calculates the correlation between two datasets at each lat lon point
def calculate_global_correlation_of_anomalies(data1, data2):
    correlation = np.zeros((data1.lat.size, data1.lon.size))
    for i in range(data1.lat.size):
        for j in range(data1.lon.size):
            anomalie_1 = data1[:,i,j]-data1[:,i,j].mean()
            anomalie_2 = data2[:,i,j]-data2[:,i,j].mean()
            correlation[i,j] = np.corrcoef(anomalie_1, anomalie_2)[0,1]
    return correlation

In [110]:
# Calculate the correlation between the sw clear sky and the surface temperature
correlation = calculate_global_correlation_of_anomalies(sw_yearly_average, suf_temp_yearly_average)

# Add the correlation to the dataset
correlation_data = xr.DataArray(correlation, coords=[sw_yearly_average.lat, sw_yearly_average.lon], dims=["lat", "lon"])

In [111]:
# Plot the correlation with a colorbar and the outline of the continents on an equal area projection
fig, ax = plt.subplots(1, 1, figsize=(20,10))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
correlation_data.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="coolwarm", cbar_kwargs={"label": "Correlation"})
plt.title("Correlation between SW Clear Sky and Surface Temperature")

# Set the latitude range from 0 to 90
ax.set_extent([-180, 180, 0, 90], crs=ccrs.PlateCarree())
plt.show()

In [112]:
# Plot the correlation with a colorbar and the outline of the continents on an equal area projection
fig, ax = plt.subplots(1, 1, figsize=(20,10))
ax = plt.axes(projection=ccrs.Mollweide())
ax.coastlines()
# Plot the correlation with a colorbar and the outline of the continents on a circular projection
correlation_data.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="coolwarm", cbar_kwargs={"label": "Correlation"})

plt.title("Correlation between SW Clear Sky and Surface Temperature")
# Set the latitude range from 0 to 90
#ax.set_extent([-180, 180, 0, 90], crs=ccrs.PlateCarree())

# Add latitude lines and labels
ax.gridlines()

In [113]:
# Calculate the average correlation at each latitude
average_correlation = correlation_data.mean(dim="lon")
# Plot the average correlation as a function of latitude
fig, ax = plt.subplots(1, 1, figsize=(20,10))
average_correlation.plot()